In [ ]:
import os, json
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import autocast
from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import f1_score, accuracy_score, classification_report

import transformers.utils.hub as _hub
_hub.list_repo_templates = lambda *a, **kw: []

from transformers import BlipProcessor, BlipForImageTextRetrieval

In [ ]:
BASE_INPUT  = "/kaggle/input/datasets/youssefelghandour11/full-dataset"
IMAGES_ROOT = f"{BASE_INPUT}/images"
VAL_LABELS  = f"{BASE_INPUT}/merged_balanced/val.json"
VAL_META    = f"{BASE_INPUT}/metadata/val.json"

# Path to the checkpoint — add it as a Kaggle dataset input or copy to /kaggle/working/
CHECKPOINT  = "/kaggle/working/blip_itm_finetuned_best.pt"

MODEL_NAME  = "Salesforce/blip-image-text-matching-base"
BATCH_SIZE  = 32
NUM_WORKERS = 4

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
def build_image_path(raw_path):
    stripped = raw_path.removeprefix("visual_news/")
    return os.path.join(IMAGES_ROOT, stripped)


def load_val_split():
    with open(VAL_LABELS) as f:
        annotations = json.load(f)["annotations"]
    with open(VAL_META) as f:
        metadata = json.load(f)

    samples, missing_meta, missing_file = [], 0, 0
    for ann in annotations:
        key = str(ann["image_id"])
        if key not in metadata:
            missing_meta += 1
            continue
        meta     = metadata[key]
        img_path = build_image_path(meta["image_path"])
        if not os.path.exists(img_path):
            missing_file += 1
            continue
        samples.append({
            "image_path": img_path,
            "caption":    meta.get("caption", ""),
            "label":      int(ann["falsified"]),
        })

    print(f"[VAL] loaded={len(samples)} | missing_meta={missing_meta} | missing_file={missing_file}")
    return samples


class ITMDataset(Dataset):
    def __init__(self, samples, processor, max_text_len=128):
        self.samples      = samples
        self.processor    = processor
        self.max_text_len = max_text_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        try:
            image = Image.open(s["image_path"]).convert("RGB")
        except (UnidentifiedImageError, OSError):
            image = Image.new("RGB", (384, 384))

        encoding = self.processor(
            images=image,
            text=s["caption"],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(s["label"], dtype=torch.long)
        return item

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  GPUs: {torch.cuda.device_count()}")

processor = BlipProcessor.from_pretrained(MODEL_NAME)
model     = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME)

state_dict = torch.load(CHECKPOINT, map_location="cpu")
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print("Checkpoint loaded.")

In [ ]:
val_samples = load_val_split()
val_ds      = ITMDataset(val_samples, processor)
val_loader  = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

all_preds, all_labels = [], []

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        pixel_values   = batch["pixel_values"].to(device)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"]

        with autocast():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_itm_head=True,
            )

        preds = outputs.itm_score.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

        if (i + 1) % 20 == 0:
            print(f"  batch {i+1}/{len(val_loader)}")

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="binary", zero_division=0)

print(f"\nVal Accuracy : {acc:.4f}")
print(f"Val F1 (bin) : {f1:.4f}")
print()
print(classification_report(all_labels, all_preds,
                             target_names=["Real", "Fake"], zero_division=0))